## Lab Assignment 02

Load spam.csv

In [1]:
import pandas as pd

df_spam = pd.read_csv(
    'spam.csv',
    encoding="latin-1"
)

# Keep the label and message columns; discard any extra columns
df_spam = df_spam.iloc[:, :2].copy()
df_spam.columns = ["label", "message"]
df_spam = df_spam.dropna(subset=["label", "message"])

df_spam["label"] = df_spam["label"].astype(str).str.strip().str.lower()
df_spam["message"] = df_spam["message"].astype(str)

print(df_spam.head())
print(df_spam["label"].value_counts())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


Make one stratified train/test split

In [2]:
from sklearn.model_selection import train_test_split

X = df_spam["message"]
y = df_spam["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Train and evaluate both models

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

models = {
    "CountVectorizer": make_pipeline(
        CountVectorizer(stop_words="english"),
        MultinomialNB()
    ),
    "TF-IDF": make_pipeline(
        TfidfVectorizer(stop_words="english"),
        MultinomialNB()
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Feature method": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Spam precision": precision_score(
            y_test, predictions, pos_label="spam", zero_division=0
        ),
        "Spam recall": recall_score(
            y_test, predictions, pos_label="spam", zero_division=0
        ),
        "Spam F1": f1_score(
            y_test, predictions, pos_label="spam", zero_division=0
        )
    })

    print(f"\n{name} classification report:")
    print(classification_report(y_test, predictions, zero_division=0))

results_df = pd.DataFrame(results).sort_values(
    by="Spam F1", ascending=False
)

print("\nModel comparison:")
print(results_df.to_string(index=False))


CountVectorizer classification report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115


TF-IDF classification report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115


Model comparison:
 Feature method  Accuracy  Spam precision  Spam recall  Spam F1
CountVectorizer  0.983857        0.958042     0.919463 0.938356
         TF-IDF  0.968610        1.000000     0.765101 0.866920


Identify which feature method performed best

In [4]:
best_method = results_df.iloc[0]["Feature method"]

print("Best method by Spam F1:", best_method)

Best method by Spam F1: CountVectorizer


1. Multinomial Naive Bayes with CountVectorizer
I trained a Multinomial Naive Bayes classifier on the SMS dataset using CountVectorizer with English stop words removed. On the test set, it achieved 98.39% accuracy. For spam messages, precision was 95.80%, recall was 91.95%, and the F1-score was 93.84%.

2. Multinomial Naive Bayes with TF-IDF
Using TF-IDF with English stop words removed, the model achieved 96.86% accuracy. For spam, precision was 100%, recall was 76.51%, and the F1-score was 86.69%.

3. Comparison and conclusion
CountVectorizer performed better overall: it had higher accuracy and a higher spam F1-score. TF-IDF had perfect spam precision, meaning that messages it flagged as spam were spam in this test set. However, its lower recall means it missed more actual spam messages.

CountVectorizer is the better feature method for this dataset, especially when the goal is to detect more spam while maintaining strong overall accuracy. If Assignment 2 used CountVectorizer as its model, these CountVectorizer results are the comparison baseline; include Assignment 2’s results alongside them if its model or test split was different.